# Steering Claude with a System Prompt

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

In [ ]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

In [ ]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

`add_user_message` and `add_assistant_message` append **role-tagged turns** to a running `messages` list. The new piece is `chat`: it takes an optional `system` argument and, when present, passes it to `client.messages.create()` as the top-level **`system`** parameter. Note that `system` is its own request field, **separate from the `messages` list** - it is not another turn in the conversation.

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Steer behavior with a system prompt

A **system prompt** sets Claude's **persona, rules, and constraints** before the conversation starts. It shapes *how* Claude responds, while the `messages` list carries *what* the user actually says. Keeping the two separate is the point: the same user question produces very different behavior depending on the system prompt in force.

Here the system prompt casts Claude as a **patient math tutor** that refuses to hand over the answer and instead **asks guiding questions**. The user gives a solvable equation, yet the response should be a Socratic nudge, not `x = 4`.

| Request field | Carries | Role |
| --- | --- | --- |
| `system` | Persona, rules, tone | Steers *how* Claude answers |
| `messages` | The back-and-forth turns | Supplies *what* was said |

> A system prompt is instruction, not dialogue. It never appears as a `user` or `assistant` turn - it is the top-level `system` parameter.

In [ ]:
system_prompt = """
You are a patient math tutor. Do not directly answer the user's question. Instead, ask follow-up questions to help the user solve the problem.
"""

messages = []

add_user_message(messages, "I need to solve the equation 2x + 3 = 11. Can you help me?")

response = chat(messages, system=system_prompt)

add_assistant_message(messages, response)

messages

[{'role': 'user',
  'content': 'I need to solve the equation 2x + 3 = 11. Can you help me?'},
 {'role': 'assistant',
  'content': "I'd be happy to help you work through this! Let me ask you some guiding questions.\n\n**First:** What do you think your goal is when solving an equation like this? What are you trying to find?\n\n**Second:** Looking at the equation 2x + 3 = 11, what operations are being done to x on the left side? (List them in the order they happen)\n\nOnce you think about these, we can work through the steps together!"}]